# My Content

On veut proposer **cinq articles par utilisateur**. Je compare le filtrage collaboratif, le filtrage par contenu et un classement par popularité qui sert de référence. Pour choisir la méthode du prototype, je privilégie la possibilité de recommander des articles sans clic.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.utils.capture import capture_output

ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data import load_data, prepare_catalog, time_periods
from src.evaluate import choose_method, evaluate, evaluation_events
from src.experiment import comparative_examples, save_results
from src.recommender import Recommender
from src.train import fit_models

## 1. Préparer les données

Le jeu Globo.com fournit les clics, le catalogue et des embeddings, qui sont des vecteurs représentant le contenu des articles. Dans l'interface, les articles sont identifiés par leur numéro.

Je retire les doublons, les lignes incomplètes et les clics datés avant la publication ou hors de leur fenêtre de session de 24 heures.

In [2]:
clicks, metadata, embeddings, quality = load_data(ROOT / "data/brut")
periods = time_periods(clicks)

print(f"{len(clicks):,} clics retenus sur {quality['raw_clicks']:,}")
print(f"{quality['outside_24h_session']:,} clics hors session et {quality['before_publication']:,} avant publication")
print(f"{clicks.user_id.nunique():,} utilisateurs et {len(metadata):,} articles dans le catalogue")
display(clicks.head(3))

2,977,561 clics retenus sur 2,988,181
7,331 clics hors session et 3,289 avant publication
322,758 utilisateurs et 364,047 articles dans le catalogue


,user_id,click_article_id,click_timestamp
0,0,157541,1506826828020
1,0,68866,1506826858020
2,0,96755,1507340033155


## 2. Comprendre les modèles

| Approche | Comment elle classe les articles | Avantage et limite |
|---|---|---|
| Collaboratif | Deux articles se rapprochent quand ils ont des lecteurs en commun. On garde jusqu'à 50 voisins par article et on utilise les dix dernières lectures pour suggérer leurs voisins. | Peut proposer des sujets différents. Un article sans clic ne peut pas avoir de voisin appris. |
| Contenu | L'ACP réduit les embeddings à 64 dimensions. On construit le profil avec les dix dernières lectures, en donnant plus de poids aux dernières, puis on classe les articles par similarité cosinus. | Peut proposer un article sans clic s'il a un embedding. Des sujets similaires peuvent dominer la liste. |
| Popularité | On trie par nombre de lecteurs, puis par date de publication en cas d'égalité. | Sert de repère et de solution de repli quand il manque des suggestions personnalisées. |

Sources des méthodes et détails des calculs dans [la comparaison détaillée](../docs/comparaison.md).

## 3. Comparer sur les mêmes clics

Les 48 dernières heures de consultation servent au **test**. Les 48 heures précédentes servent à la **validation**, où je choisis le modèle. Toutes les six heures, les modèles apprennent avec les clics antérieurs à la fenêtre. L'ACP est ajustée avant la validation puis reste fixe.

Pour chaque période, je prends 2 000 cas de façon reproductible. Pour un utilisateur déjà connu, on cherche à retrouver son premier clic vers un article inédit dans la fenêtre. Les méthodes partagent les mêmes candidats, déjà publiés à la date du clic et non lus à l'entraînement. On cherche d'abord dans les sept derniers jours, puis on élargit selon le nombre de candidats disponibles.

Le **Hit Rate@5** indique la part des cas où l'article cliqué est dans les cinq suggestions. Le **MRR@5** et le **NDCG@5** donnent plus de poids aux premières places.

In [3]:
catalog, variance = prepare_catalog(metadata, embeddings, clicks, periods["validation_start"])

names = {"collaborative": "Collaboratif", "content": "Contenu", "popular": "Popularité"}
columns = {
    "hit_rate_at_5": "Hit Rate@5 (%)",
    "mrr_at_5": "MRR@5",
    "ndcg_at_5": "NDCG@5",
    "distinct_categories": "Catégories par liste",
    "personalized_new_article_share": "Suggestions personnalisées sans clic (%)",
}


def results_table(evaluation):
    table = pd.DataFrame(evaluation["methods"]).T[list(columns)].astype(float)
    table[["hit_rate_at_5", "personalized_new_article_share"]] *= 100
    return table.rename(index=names, columns=columns).round(4)


events, validation_count = evaluation_events(clicks, periods["validation_start"], periods["test_start"])
with capture_output():
    validation, validation_details = evaluate(clicks, catalog, events)
decision = choose_method(validation, require_new_articles=True)
display(results_table(validation))
print("Choix avant le test", names[decision["selected_method"]])

,Hit Rate@5 (%),MRR@5,NDCG@5,Catégories par liste,Suggestions personnalisées sans clic (%)
Collaboratif,2.10,0.0118,0.0140,4.1990,0.00
Contenu,1.50,0.0071,0.0090,2.2075,4.31
Popularité,1.05,0.0026,0.0045,3.7475,0.00


Choix avant le test Contenu


In [4]:
events, test_count = evaluation_events(clicks, periods["test_start"], periods["end"])
with capture_output():
    test, test_details = evaluate(clicks, catalog, events)

print(f"Test sur {test['cases']:,} cas")
display(results_table(test))

Test sur 2,000 cas


,Hit Rate@5 (%),MRR@5,NDCG@5,Catégories par liste,Suggestions personnalisées sans clic (%)
Collaboratif,2.00,0.0106,0.0130,4.201,0.0
Contenu,1.25,0.0062,0.0078,2.221,5.1
Popularité,1.60,0.0040,0.0069,4.269,0.0


Le **collaboratif** retrouve davantage de prochains clics. Je retiens le **contenu** parce qu'il peut aussi personnaliser les suggestions d'articles sans clic à partir de leurs embeddings. Sur ce jeu historique, les deux scores de pertinence restent faibles.

## 4. Trois cas à expliquer

- **Diversité.** Pour l'utilisateur 1956, le contenu propose cinq articles d'une seule catégorie. Le collaboratif propose cinq catégories. Ce cas illustre un risque de répétition des sujets. Mesurer un effet de bulle demanderait de suivre les lectures dans le temps.
- **Article sans clic.** L'article 202652 n'avait pas encore été consulté à l'entraînement. Le contenu le retrouve au rang 4. C'est le clic qui a suivi dans cet exemple de validation.
- **Nouvel utilisateur.** Sans historique, les deux méthodes utilisent la même sélection de repli. Après un premier article apprécié, le contenu peut créer un profil. Le collaboratif peut personnaliser si cet article a des voisins parmi les candidats. Dans le cas ci-dessous, les candidats de la date simulée n'ont pas de clics d'apprentissage.

In [5]:
with capture_output():
    model = fit_models(clicks, catalog)
scenarios = comparative_examples(model, clicks, validation_details)

concentration = scenarios["concentration"]
display(pd.DataFrame({
    "Catégories du contenu": concentration["content_categories"],
    "Catégories du collaboratif": concentration["collaborative_categories"],
}, index=range(1, 6)))

cold = scenarios["new_article"]
if cold:
    print(f"Article {cold['article_id']} sans clic, rang {cold['content_rank']} avec le contenu")
    print("Dans les cinq du collaboratif ?", cold["article_id"] in cold["collaborative_articles"])

new_user = scenarios["new_user"]
display(pd.DataFrame({
    "Sans préférence (repli sur 5)": {
        names[method]: result["fallback_count"] for method, result in new_user["without_history"].items()
    },
    "Après une préférence (repli sur 5)": {
        names[method]: result["fallback_count"] for method, result in new_user["with_first_preference"].items()
    },
}))

,Catégories du contenu,Catégories du collaboratif
1,412,399
2,412,118
3,412,354
4,412,209
5,412,412


Article 202652 sans clic, rang 4 avec le contenu
Dans les cinq du collaboratif ? False


,Sans préférence (repli sur 5),Après une préférence (repli sur 5)
Collaboratif,5,5
Contenu,5,0


## 5. Exporter et vérifier les cinq suggestions

Streamlit charge le modèle et affiche cinq articles distincts, déjà publiés et non lus. La date de la dernière création ou consultation sert d'« aujourd'hui » dans la démonstration. Les dates de validation et de test restent historiques.

L'export peut être utilisé par la future Azure Function. Le [schéma cible et les mises à jour des bases](../docs/architecture.md) et le [suivi du barème](../docs/bareme.md) sont documentés à part.

In [6]:
output = ROOT / "models"
save_results(
    output, model, quality, periods, variance, validation, test, decision,
    validation_count, test_count, validation_details + test_details, scenarios, sample=False,
)
loaded = Recommender.load(output / "recommender.npz")
for method in ("collaborative", "content"):
    result = loaded.recommend(1956, method)
    assert len(set(result["article_ids"])) == 5
    assert set(result["article_ids"]).isdisjoint(loaded.history(1956))
    assert all(article["published_at"] <= loaded.reference_time for article in result["articles"])
    print(names[method], result["article_ids"])

Collaboratif [39891, 263284, 347438, 70039, 342184]
Contenu [36360, 212641, 156767, 159718, 157025]
